# 04 — Tiny CNN → ResNet block

We'll stack the conv2d from NB 03 into a real (tiny) CNN, train it on CIFAR-10, then add a **residual connection** and see why ResNet50 can stack 50 layers deep without falling apart.

## Roadmap
1. Multi-channel conv (RGB in, many feature maps out) — the actual building block
2. Build a tiny CNN: 2 conv blocks + FC head
3. Why deep nets break: the **vanishing gradient** problem (demonstrated)
4. The **residual block** — 5 lines of code, decade-defining idea
5. How ResNet50 scales this to 50 layers

We'll use NumPy for forward passes and gradient bookkeeping but lean on PyTorch's `autograd` for the actual gradients here — implementing multi-channel conv backprop in NumPy is 200 lines of index-arithmetic and obscures the *idea*. This is the right time to introduce why PyTorch exists.


In [ ]:
# === Bootstrap (safe to re-run) ===
# Installs minimal deps and downloads tiny CIFAR-10 subset for any notebook
# that needs it. The from-scratch notebooks deliberately avoid importing the
# project package so they run anywhere (Colab, plain Python, Jupyter).
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "numpy", "matplotlib", "pillow", "torchvision"], check=True)
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)
print("numpy", np.__version__)


## 1. Multi-channel conv — the real building block

The conv from NB 03 was single-channel. A real conv layer has:

- **Input:** `(C_in, H, W)` — e.g. RGB image: C_in=3
- **Kernel:** `(C_out, C_in, kH, kW)` — `C_out` filters, each spanning all `C_in` input channels
- **Output:** `(C_out, H', W')`

Each output channel is the sum of separately-convolving each input channel with its own kernel slice, plus a bias.


In [ ]:
import numpy as np
def multi_channel_conv2d(x, W, b, padding=1):
    """x: (C_in,H,W); W: (C_out,C_in,k,k); b: (C_out,)"""
    C_in, H, Wd = x.shape
    C_out, _, k, _ = W.shape
    xp = np.pad(x, [(0,0),(padding,padding),(padding,padding)])
    Hout = H + 2*padding - k + 1
    Wout = Wd + 2*padding - k + 1
    out = np.zeros((C_out, Hout, Wout), dtype=np.float32)
    for o in range(C_out):
        for y in range(Hout):
            for X in range(Wout):
                patch = xp[:, y:y+k, X:X+k]
                out[o, y, X] = (patch * W[o]).sum() + b[o]
    return out

x = np.random.randn(3, 8, 8).astype(np.float32)
W = np.random.randn(16, 3, 3, 3).astype(np.float32) * 0.1
b = np.zeros(16, dtype=np.float32)
y = multi_channel_conv2d(x, W, b, padding=1)
print("input:", x.shape, " → output:", y.shape, "  (3 RGB → 16 feature maps)")


## 2. Tiny CNN architecture

Two conv blocks (conv → ReLU → pool), then flatten, then a dense head. Small enough to train on CPU in minutes.

```
(3, 32, 32) → conv(16) → ReLU → maxpool 2     → (16, 16, 16)
            → conv(32) → ReLU → maxpool 2     → (32,  8,  8)
            → flatten                          → (2048,)
            → dense(10)                        → (10,)
            → softmax + cross-entropy
```

For training I'll switch to PyTorch (autograd) but the architecture is exactly what we built above — same operations, PyTorch just handles the gradient bookkeeping.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class TinyCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool  = nn.MaxPool2d(2, 2)
        self.fc    = nn.Linear(32 * 8 * 8, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))     # (B, 16, 16, 16)
        x = self.pool(F.relu(self.conv2(x)))     # (B, 32,  8,  8)
        x = x.flatten(1)                         # (B, 2048)
        return self.fc(x)                         # (B, 10)

print(TinyCNN())
print("param count:", sum(p.numel() for p in TinyCNN().parameters()))


In [ ]:
# Load a tiny CIFAR-10 subset.
# torchvision is used ONLY to fetch the dataset bytes — no torch tensors,
# no torch.nn. We immediately convert to NumPy.
from torchvision import datasets
from pathlib import Path
DATA_ROOT = Path("./_cifar_cache")
DATA_ROOT.mkdir(exist_ok=True)

train_ds = datasets.CIFAR10(root=DATA_ROOT, train=True,  download=True)
test_ds  = datasets.CIFAR10(root=DATA_ROOT, train=False, download=True)

def to_numpy(ds, n):
    X = np.stack([np.asarray(ds[i][0], dtype=np.float32) / 255.0 for i in range(n)])
    y = np.array([ds[i][1] for i in range(n)], dtype=np.int64)
    return X, y

X_train, y_train = to_numpy(train_ds, 2000)
X_test,  y_test  = to_numpy(test_ds,  500)
CLASSES = ("plane","car","bird","cat","deer","dog","frog","horse","ship","truck")
print("X_train", X_train.shape, "y_train", y_train.shape)
print("X_test ", X_test.shape,  "y_test ", y_test.shape)


In [ ]:
# Train the TinyCNN for a few epochs. Pure-NumPy training would take much longer
# because of conv backprop; that 200-line implementation is what `autograd` saves.
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

def to_chw(X):  # (N,H,W,C) → (N,C,H,W)
    return np.transpose(X, (0, 3, 1, 2))

Xt = torch.tensor(to_chw(X_train)).float()
yt = torch.tensor(y_train)
Xv = torch.tensor(to_chw(X_test)).float()
yv = torch.tensor(y_test)

model = TinyCNN().to(device)
optim = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0.9)
crit  = nn.CrossEntropyLoss()

BS = 64
loss_hist, acc_hist = [], []
for ep in range(8):
    model.train()
    perm = torch.randperm(len(Xt))
    for i in range(0, len(Xt), BS):
        idx = perm[i:i+BS]
        xb, yb = Xt[idx].to(device), yt[idx].to(device)
        optim.zero_grad()
        loss = crit(model(xb), yb)
        loss.backward()
        optim.step()
    model.eval()
    with torch.no_grad():
        preds = model(Xv.to(device)).argmax(dim=-1).cpu()
    acc = (preds == yv).float().mean().item()
    loss_hist.append(loss.item()); acc_hist.append(acc)
    print(f"epoch {ep+1}  loss={loss.item():.3f}  test_acc={acc:.3f}")

import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(loss_hist); ax[0].set_title("train loss"); ax[0].grid(True)
ax[1].plot(acc_hist);  ax[1].set_title("test acc");   ax[1].grid(True)
plt.show()


## 3. The vanishing-gradient problem

Naively stacking 50 conv layers does **not** work. Each layer applies its weight matrix during the forward pass, and during backprop you multiply by all those weights again — small weights compound to ~zero gradient at the early layers; large weights compound to NaN. Training the early layers stops working.

Let me show this concretely by stacking many tanh activations and watching the gradient magnitude collapse.


In [ ]:
import numpy as np
np.random.seed(0)

# Toy network: chain of L tanh activations, each fed through a random weight matrix.
def gradient_norm_after_L_layers(L, dim=64):
    x = np.random.randn(dim)
    grad = np.ones(dim)  # gradient at the output
    for _ in range(L):
        W = np.random.randn(dim, dim) * 0.5  # smallish weights
        z = W @ x
        x = np.tanh(z)
        # backward: ∂tanh/∂z = 1 - tanh²
        grad = (1 - x**2) * (W.T @ grad)
    return float(np.linalg.norm(grad))

depths = [1, 5, 10, 20, 50]
for L in depths:
    print(f"L={L:3d}: ||grad|| at input = {gradient_norm_after_L_layers(L):.3e}")


You'll see the gradient norm collapse exponentially — by 50 layers it's effectively zero. The first layer has no signal to learn from.

## 4. The residual fix

In each block, instead of computing $y = F(x)$, compute

$$y = F(x) + x$$

The "+x" is the **skip connection**. The block now learns the *residual* $F(x)$ — the change you want to add to $x$.

Two benefits:

1. **Identity is free.** If the best $F$ is "do nothing," the network drives $F(x) \to 0$ trivially.
2. **Gradient highway.** $\partial y / \partial x = \partial F / \partial x + 1$. That `+1` means gradient *never* fully vanishes — it has a direct path back through every skip.


In [ ]:
import torch.nn as nn

class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)

    def forward(self, x):
        identity = x
        out = F.relu(self.conv1(x))
        out = self.conv2(out)
        return F.relu(out + identity)   # ← the entire trick

# Demo: same gradient experiment, now with residual connections
import torch
def grad_norm_residual(L, dim=64):
    x = torch.randn(1, dim, 4, 4, requires_grad=True)
    blocks = nn.Sequential(*[ResidualBlock(dim) for _ in range(L)])
    y = blocks(x).sum()
    y.backward()
    return float(x.grad.norm())

# Disable gradient warning for the demo
import warnings; warnings.filterwarnings("ignore")
for L in [1, 5, 10, 20, 50]:
    print(f"residual L={L:3d}: ||grad|| = {grad_norm_residual(L):.3e}")


Notice the gradient norm stays in a sensible range even at 50 layers. *That* is why ResNet50 can train end-to-end where a 50-layer plain CNN cannot.

## 5. How ResNet50 scales the idea

ResNet50 = 50 weight layers organised as:

- 1 initial conv (7×7, stride 2)
- 4 stages of residual blocks (3, 4, 6, 3 blocks each)
- Each block is `1×1 conv → 3×3 conv → 1×1 conv` ("bottleneck") + skip
- Global average pool → FC(1000)

Same residual idea, repeated. No new math.

**Next:** the totally different problem of **object detection** — predicting *where* things are, not just what.
